# B2 · Tokenisation et embeddings — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b2_tokenisation/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b2_tokenisation/03_notebook.ipynb`, créée par `python tools/start_chapter.py B2`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B2.Q1 à B2.Q12, B2.R1 à B2.R3, B2.1 à B2.6, B2.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | B2.13 à B2.14 | des tokenizers réels sur Holmes et Verne, et ce que coûtent les langues, les nombres et les emoji (*prochaine session de génération*) | | | |
| B | B2.15 à B2.19 | ton BPE : compter et fusionner les paires, apprendre les fusions, encoder et décoder, réparer un tokenizer qui abîme les accents, te comparer à la bibliothèque tokenizers (*prochaine session de génération*) | | | |
| C | B2.20 à B2.21 | expériences : compression et entropie selon la taille du vocabulaire, tokenizer d'une langue appliqué à l'autre (*prochaine session de génération*) | | | |
| D | B2.22 à B2.27 | vecteurs de mots : co-occurrences et PPMI, cosinus et voisins, PPMI + SVD sur Holmes, word2vec sur Verne, voisins de « Fogg » et « Passepartout », carte des embeddings (*prochaine session de génération*) | | | |
| E | B2.28 à B2.31 | embeddings de phrases et recherche sémantique, statique contre contextuel, sauvegarder un tokenizer, et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")  # your own library

## Objectifs et rappel express

- Manipuler Unicode, UTF-8 et la normalisation d'un texte, et compter ses octets et ses points de code.
- Écrire un tokenizer BPE (entraînement, encodage par rang, décodage) et le comparer à la bibliothèque `tokenizers`.
- Comparer des tokenizers réels et mesurer leur coût (fertilité, compression) selon la langue.
- Relier `nn.Embedding` à un produit one-hot × table, et entraîner des embeddings word2vec.
- Construire des embeddings par co-occurrences (PPMI et SVD), les interroger par le cosinus, et chercher des passages par des embeddings de phrases.

**Rappel express.** Fertilité $= \frac{\text{tokens}}{\text{mots}}$, compression $= \frac{\text{octets}}{\text{tokens}}$. Fusion BPE : la paire voisine la plus fréquente (à égalité, la plus petite paire d'identifiants) devient un token ; on encode en appliquant les fusions par rang. $\mathbf{x}_{\text{one-hot}}^\top \mathbf{E} = \mathbf{E}[i, :]$. $\operatorname{PPMI}(w, c) = \max\big(0, \log \frac{p(w, c)}{p(w)\,p(c)}\big)$. Skip-gram : $L = -\log \sigma(\mathbf{u}_c^\top \mathbf{v}_w) - \sum_k \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_w)$. Mean pooling : $\mathbf{s} = \frac{\sum_t m_t \mathbf{h}_t}{\sum_t m_t}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B2.1 à B2.6, 🧮 B2.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un mot : entre guillemets (`"mot"`) ; des octets : une liste, en hexadécimal ou en décimal (`[0x4E, 0x6F]` ou `[78, 111]`) ; des identifiants, une paire d'identifiants ou un vecteur : une liste (`[4, 0, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B2.7, l'estimation 🧮 B2.8, la réflexion (🗣️ B2.10, ⚖️ B2.11, 📄 B2.12) et l'entretien se corrigent avec `05_solutions.md`.

**B2.Q1 — Caractères, mots ou sous-mots : taille du vocabulaire contre longueur des séquences**

In [ ]:
# B2.Q1: the values you found on paper
answer_B2_Q1a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q1b = ...  # b) True or False
answer_B2_Q1c = ...  # c) a whole number
answer_B2_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q1a, answer_B2_Q1b, answer_B2_Q1c, answer_B2_Q1d, answer_B2_Q1e]):
    wb.check(f"B2.Q1{letter}", answer)

**B2.Q2 — Octets, points de code, caractères affichés : combien « é » compte-t-il ?**

In [ ]:
# B2.Q2: the values you found on paper
answer_B2_Q2a = ...  # a) a whole number
answer_B2_Q2b = ...  # b) a whole number
answer_B2_Q2c = ...  # c) a whole number
answer_B2_Q2d = ...  # d) a whole number
answer_B2_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q2a, answer_B2_Q2b, answer_B2_Q2c, answer_B2_Q2d, answer_B2_Q2e]):
    wb.check(f"B2.Q2{letter}", answer)

**B2.Q3 — NFC ou NFD : deux chaînes identiques à l'écran, différentes en mémoire**

In [ ]:
# B2.Q3: the values you found on paper
answer_B2_Q3a = ...  # a) True or False
answer_B2_Q3b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q3c = ...  # c) True or False
answer_B2_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q3a, answer_B2_Q3b, answer_B2_Q3c, answer_B2_Q3d, answer_B2_Q3e]):
    wb.check(f"B2.Q3{letter}", answer)

**B2.Q4 — Fusions BPE : pourquoi la liste ordonnée des fusions est le modèle**

In [ ]:
# B2.Q4: the values you found on paper
answer_B2_Q4a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q4b = ...  # b) True or False
answer_B2_Q4c = ...  # c) a whole number
answer_B2_Q4d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q4a, answer_B2_Q4b, answer_B2_Q4c, answer_B2_Q4d, answer_B2_Q4e]):
    wb.check(f"B2.Q4{letter}", answer)

**B2.Q5 — BPE au niveau octet : jamais de token inconnu, vrai ou faux ?**

In [ ]:
# B2.Q5: the values you found on paper
answer_B2_Q5a = ...  # a) True or False
answer_B2_Q5b = ...  # b) a whole number
answer_B2_Q5c = ...  # c) True or False
answer_B2_Q5d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q5a, answer_B2_Q5b, answer_B2_Q5c, answer_B2_Q5d, answer_B2_Q5e]):
    wb.check(f"B2.Q5{letter}", answer)

**B2.Q6 — Tokens spéciaux et espace initial : « Ġthe » n'est pas « the »**

In [ ]:
# B2.Q6: the values you found on paper
answer_B2_Q6a = ...  # a) True or False
answer_B2_Q6b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_Q6d = ...  # d) True or False
answer_B2_Q6e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B2_Q6a, answer_B2_Q6b, answer_B2_Q6c, answer_B2_Q6d, answer_B2_Q6e]):
    wb.check(f"B2.Q6{letter}", answer)

**B2.Q7 — WordPiece, Unigram, SentencePiece : quel critère pour quelle méthode ?**

In [ ]:
# B2.Q7: the values you found on paper
answer_B2_Q7a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q7b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q7c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_Q7d = ...  # d) True or False
answer_B2_Q7e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B2_Q7a, answer_B2_Q7b, answer_B2_Q7c, answer_B2_Q7d, answer_B2_Q7e]):
    wb.check(f"B2.Q7{letter}", answer)

**B2.Q8 — Pourquoi un LLM compte mal les lettres d'un mot et calcule mal avec les chiffres**

In [ ]:
# B2.Q8: the values you found on paper
answer_B2_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q8b = ...  # b) True or False
answer_B2_Q8c = ...  # c) a whole number
answer_B2_Q8d = ...  # d) a whole number
answer_B2_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q8a, answer_B2_Q8b, answer_B2_Q8c, answer_B2_Q8d, answer_B2_Q8e]):
    wb.check(f"B2.Q8{letter}", answer)

**B2.Q9 — nn.Embedding : une table de consultation entraînable, pas une couche magique**

In [ ]:
# B2.Q9: the values you found on paper
answer_B2_Q9a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B2_Q9b = ...  # b) a whole number
answer_B2_Q9c = ...  # c) True or False
answer_B2_Q9d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q9a, answer_B2_Q9b, answer_B2_Q9c, answer_B2_Q9d, answer_B2_Q9e]):
    wb.check(f"B2.Q9{letter}", answer)

**B2.Q10 — L'hypothèse distributionnelle : un mot se définit par ses voisins**

In [ ]:
# B2.Q10: the values you found on paper
answer_B2_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q10b = ...  # b) True or False
answer_B2_Q10c = ...  # c) a number, 2 decimals
answer_B2_Q10d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q10a, answer_B2_Q10b, answer_B2_Q10c, answer_B2_Q10d, answer_B2_Q10e]):
    wb.check(f"B2.Q10{letter}", answer)

**B2.Q11 — Statique ou contextuel : l'« avocat » du tribunal et celui du marché**

In [ ]:
# B2.Q11: the values you found on paper
answer_B2_Q11a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q11b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q11c = ...  # c) True or False
answer_B2_Q11d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q11e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_Q11a, answer_B2_Q11b, answer_B2_Q11c, answer_B2_Q11d, answer_B2_Q11e]):
    wb.check(f"B2.Q11{letter}", answer)

**B2.Q12 — Cosinus ou distance euclidienne pour des embeddings normalisés ?**

In [ ]:
# B2.Q12: the values you found on paper
answer_B2_Q12a = ...  # a) a number, 1 decimal
answer_B2_Q12b = ...  # b) True or False
answer_B2_Q12c = ...  # c) a number, 3 decimals
answer_B2_Q12d = ...  # d) True or False
answer_B2_Q12e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_Q12a, answer_B2_Q12b, answer_B2_Q12c, answer_B2_Q12d, answer_B2_Q12e]):
    wb.check(f"B2.Q12{letter}", answer)

**B2.R1 — À quoi sert l'embedding d'image extrait d'un réseau pré-entraîné ?**

In [ ]:
# B2.R1: the values you found on paper
answer_B2_R1a = ...  # a) True or False
answer_B2_R1b = ...  # b) a whole number
answer_B2_R1c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_R1d = ...  # d) True or False
answer_B2_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B2_R1a, answer_B2_R1b, answer_B2_R1c, answer_B2_R1d, answer_B2_R1e]):
    wb.check(f"B2.R1{letter}", answer)

**B2.R2 — build_vocab, encode, decode et le token inconnu**

In [ ]:
# B2.R2: the values you found on paper
answer_B2_R2a = ...  # a) a whole number
answer_B2_R2b = ...  # b) a list of token ids, e.g. [4, 0, 7]
answer_B2_R2c = ...  # c) True or False
answer_B2_R2d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B2_R2a, answer_B2_R2b, answer_B2_R2c, answer_B2_R2d, answer_B2_R2e]):
    wb.check(f"B2.R2{letter}", answer)

**B2.R3 — Entropie, Huffman et ce que l'on compresse vraiment**

In [ ]:
# B2.R3: the values you found on paper
answer_B2_R3a = ...  # a) a number, 2 decimals
answer_B2_R3b = ...  # b) a whole number
answer_B2_R3c = ...  # c) True or False
answer_B2_R3d = ...  # d) True or False
answer_B2_R3e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_R3a, answer_B2_R3b, answer_B2_R3c, answer_B2_R3d, answer_B2_R3e]):
    wb.check(f"B2.R3{letter}", answer)

**Ex B2.1 — « Noël à Paris » en UTF-8 : octets, points de code, longueur**

In [ ]:
# Ex B2.1: the values you found on paper
answer_B2_1a = ...  # a) a whole number
answer_B2_1b = ...  # b) a whole number
answer_B2_1c = ...  # c) a list of byte values, e.g. [0x4E, 0x6F]
answer_B2_1d = ...  # d) a whole number
answer_B2_1e = ...  # e) a whole number
answer_B2_1f = ...  # f) a list of byte values, e.g. [0x4E, 0x6F]
answer_B2_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_1a, answer_B2_1b, answer_B2_1c, answer_B2_1d, answer_B2_1e, answer_B2_1f, answer_B2_1g]):
    wb.check(f"B2.1{letter}", answer)

**Ex B2.2 — Analogie à la main : roi − homme + femme sur des vecteurs de dimension 3**

In [ ]:
# Ex B2.2: the values you found on paper
answer_B2_2a = ...  # a) a number, 2 decimals
answer_B2_2b = ...  # b) a number, 3 decimals
answer_B2_2c = ...  # c) a number, 3 decimals
answer_B2_2d = ...  # d) a word as a string, e.g. "mot"
answer_B2_2e = ...  # e) a word as a string, e.g. "mot"
answer_B2_2f = ...  # f) a word as a string, e.g. "mot"
answer_B2_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_2a, answer_B2_2b, answer_B2_2c, answer_B2_2d, answer_B2_2e, answer_B2_2f, answer_B2_2g]):
    wb.check(f"B2.2{letter}", answer)

**Ex B2.3 — Trois fusions BPE à la main sur un mini-corpus**

In [ ]:
# Ex B2.3: the values you found on paper
answer_B2_3a = ...  # a) a whole number
answer_B2_3b = ...  # b) a whole number
answer_B2_3c = ...  # c) a pair of ids, e.g. (2, 7)
answer_B2_3d = ...  # d) a whole number
answer_B2_3e = ...  # e) a pair of ids, e.g. (2, 7)
answer_B2_3f = ...  # f) a pair of ids, e.g. (2, 7)
answer_B2_3g = ...  # g) a whole number
answer_B2_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_3a, answer_B2_3b, answer_B2_3c, answer_B2_3d, answer_B2_3e, answer_B2_3f, answer_B2_3g, answer_B2_3h]):
    wb.check(f"B2.3{letter}", answer)

**Ex B2.4 — Encoder un mot jamais vu avec des fusions déjà apprises**

In [ ]:
# Ex B2.4: the values you found on paper
answer_B2_4a = ...  # a) a list of token ids, e.g. [4, 0, 7]
answer_B2_4b = ...  # b) a list of token ids, e.g. [4, 0, 7]
answer_B2_4c = ...  # c) a list of token ids, e.g. [4, 0, 7]
answer_B2_4d = ...  # d) a list of token ids, e.g. [4, 0, 7]
answer_B2_4e = ...  # e) a list of token ids, e.g. [4, 0, 7]
answer_B2_4f = ...  # f) a whole number
answer_B2_4g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_4a, answer_B2_4b, answer_B2_4c, answer_B2_4d, answer_B2_4e, answer_B2_4f, answer_B2_4g]):
    wb.check(f"B2.4{letter}", answer)

**Ex B2.5 — One-hot fois matrice : nn.Embedding à la main, et quelles lignes reçoivent un gradient**

In [ ]:
# Ex B2.5: the values you found on paper
answer_B2_5a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B2_5b = ...  # b) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5c = ...  # c) a number, 1 decimal
answer_B2_5d = ...  # d) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5e = ...  # e) a whole number
answer_B2_5f = ...  # f) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5g = ...  # g) True or False
answer_B2_5h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_5a, answer_B2_5b, answer_B2_5c, answer_B2_5d, answer_B2_5e, answer_B2_5f, answer_B2_5g, answer_B2_5h]):
    wb.check(f"B2.5{letter}", answer)

**Ex B2.6 — Co-occurrences et PPMI à la main (fenêtre de 1, quatre mots)**

In [ ]:
# Ex B2.6: the values you found on paper
answer_B2_6a = ...  # a) a whole number
answer_B2_6b = ...  # b) a whole number
answer_B2_6c = ...  # c) a number, 3 decimals
answer_B2_6d = ...  # d) a number, 3 decimals
answer_B2_6e = ...  # e) a number, 3 decimals
answer_B2_6f = ...  # f) a number, 3 decimals
answer_B2_6g = ...  # g) True or False
answer_B2_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_6a, answer_B2_6b, answer_B2_6c, answer_B2_6d, answer_B2_6e, answer_B2_6f, answer_B2_6g, answer_B2_6h]):
    wb.check(f"B2.6{letter}", answer)

**Ex B2.9 — Fermi : poids de la table d'embedding de GPT-2 et d'un LLM à 128 000 tokens**

In [ ]:
# Ex B2.9: the values you found on paper
answer_B2_9a = ...  # a) a whole number
answer_B2_9b = ...  # b) a number, 1 decimal
answer_B2_9c = ...  # c) a number, 1 decimal
answer_B2_9d = ...  # d) a whole number
answer_B2_9e = ...  # e) a number, 1 decimal
answer_B2_9f = ...  # f) a whole number
answer_B2_9g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_9a, answer_B2_9b, answer_B2_9c, answer_B2_9d, answer_B2_9e, answer_B2_9f, answer_B2_9g]):
    wb.check(f"B2.9{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu compter les octets et les points de code d'un texte, et écrire un caractère en UTF-8 ?
2. Sais-tu dérouler un entraînement BPE à la main, départage compris, puis encoder un mot nouveau par rang ?
3. Sais-tu dire quelles lignes d'une table d'embedding reçoivent un gradient, et calculer une PPMI ?
4. Sais-tu chiffrer ce que coûtent une table d'embedding et un texte selon sa langue ?

**Pour aller plus loin** : le chapitre 6 du cours LLM de Hugging Face et `minbpe` d'A. Karpathy, cités dans la fiche. La suite : le bonus B3, attention et Transformers.

*Les exercices de code B2.13 à B2.31 (parties A à E : tokenizers réels, ton BPE, expériences, vecteurs de mots, embeddings de phrases, livraison et défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B2` pour obtenir le notebook complet.*